# 05 · Verified CSP/QCSP pretraining for native AHR

**Purpose.** Learn finite AHR energy preferences from verified constraint decisions, using the nested Queen/Worker/Scout architecture in 04. This is the CSP-only route: no Go examples, engine labels or Go parameter updates. The shared input codec is also consumed by 06 and 07.

**Inputs:** tagged definitions in 00–04; the explicit CSP train/dev shards in `data/starter_v1_0687ec005196/`; optional resumable CSP curriculum expansion. **Outputs:** validation and development reports under `reports/`, prepared data under `data/`, and complete training state under `checkpoints/<run_id>/`. Default **Run All performs zero optimizer updates** and should take 1–3 minutes. No pretrained model or learned improvement is claimed by validation. Actual training and the short optimizer recovery test require an explicit launch described below.

The supplied theory (physical page 80, §20.1) proposes `max(0, margin + E_positive − E_negative)`. Here each positive/negative pair refers to a verified existential decision at an observed prefix. Future universal choices remain quantified. Hard verification is outside differentiation. The neural parameterization, byte codec and finite curriculum are engineering choices, not proofs of general reasoning or Go strength.

In [ ]:
from pathlib import Path
import nbformat, os, json
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/00_instance_setup_and_recovery.ipynb").exists())
# Only tagged definitions execute; upstream setup, generation and training never run.
for filename in ["00_instance_setup_and_recovery.ipynb", "01_go_rules_and_independent_verifier.ipynb", "02_go_to_csp_and_qcsp.ipynb", "03_curricula_datasets_and_splits.ipynb", "04_native_ahr_architecture.ipynb"]:
    notebook = nbformat.read(PROJECT / "notebooks" / filename, 4)
    for index, cell in enumerate(notebook.cells):
        if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", []):
            exec(compile(cell.source, f"{filename}:cell-{index}", "exec"), globals())
print("Definitions loaded. Default validation: no optimizer updates.")

## One addressed, lossless codec for constraints and Go

Canonical UTF-8 JSON is split into 12-byte tokens. Sixteen input features hold 12 byte values, byte offset, candidate owner, actual byte count and token type. Each candidate has one action anchor; its complete payload remains visible in owned tokens. Context owner is zero. These explicit addresses matter because 04 has no positional embeddings. Anchors are ordered exactly as the supplied actions. The codec encodes no training label or certificate. Exceeding a declared bound raises `ReasoningEncodingError`; 06 must abstain or use a verified legal fallback rather than truncate superko history.

Anchors carry the last 12 payload bytes and their true offset, making the CSP action value directly visible. Candidate payloads still share the model's global role state; addresses do not provide explicit candidate-local attention. This limits binding capacity and is a hypothesis to measure, not an established universal representation.

In [ ]:
from dataclasses import dataclass, asdict
from collections import Counter
import hashlib, math, itertools, time, copy, random, socket, subprocess, sys
import numpy as np
import torch

@dataclass(frozen=True)
class ReasoningCodecConfig:
    max_tokens: int = 4096
    max_actions: int = 128
    max_payload_bytes: int = 65536
    bytes_per_token: int = 12

class ReasoningEncodingError(ValueError):
    pass

def reasoning_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False)

def reasoning_digest(value):
    return hashlib.sha256(reasoning_json(value).encode("utf-8")).hexdigest()

def _validate_reasoning_codec(config):
    if not isinstance(config, ReasoningCodecConfig):
        raise ReasoningEncodingError("Expected ReasoningCodecConfig")
    if config.bytes_per_token != 12 or not 1 <= config.max_tokens <= 65536 or not 1 <= config.max_actions <= 128 or not 1 <= config.max_payload_bytes <= 65536:
        raise ReasoningEncodingError("Codec bounds unsupported by exact float32 addresses")

def encode_reasoning_actions(context, actions, config=None, device="cpu"):
    config = config or ReasoningCodecConfig()
    _validate_reasoning_codec(config)
    if not isinstance(actions, (list, tuple)) or not 1 <= len(actions) <= config.max_actions:
        raise ReasoningEncodingError("Need 1..max_actions candidates")
    try:
        payloads = [reasoning_json(v).encode("utf-8") for v in [context, *actions]]
    except (ValueError, TypeError, UnicodeError) as error:
        raise ReasoningEncodingError("Only finite JSON values are supported") from error
    if sum(map(len, payloads)) > config.max_payload_bytes:
        raise ReasoningEncodingError("Payload capacity exceeded; no truncation")
    token_count = sum((len(b) + 11) // 12 for b in payloads) + len(actions)
    if token_count > config.max_tokens:
        raise ReasoningEncodingError("Token capacity exceeded; no truncation")
    rows, anchors = [], []
    def token(chunk, offset, owner, kind):
        values = [byte / 255.0 for byte in chunk] + [0.0] * (12 - len(chunk))
        return values + [offset / 65536.0, owner / 128.0, len(chunk) / 16.0, kind / 4.0]
    for owner, payload in enumerate(payloads):
        for offset in range(0, len(payload), 12):
            rows.append(token(payload[offset:offset+12], offset, owner, 0 if owner == 0 else 1))
            anchors.append(False)
        if owner:
            anchor_offset = max(0, len(payload) - 12)
            rows.append(token(payload[anchor_offset:], anchor_offset, owner, 2))
            anchors.append(True)
    return {"x": torch.tensor(rows, dtype=torch.float32, device=device).unsqueeze(0),
            "valid_mask": torch.ones((1, len(rows)), dtype=torch.bool, device=device),
            "action_mask": torch.tensor([anchors], dtype=torch.bool, device=device)}

def decode_reasoning_actions(batch):
    # Diagnostic inverse, also validates exact address/ownership recovery.
    pieces, anchor_owners = {}, []
    for row, valid, action in zip(batch["x"][0].detach().cpu(), batch["valid_mask"][0].cpu(), batch["action_mask"][0].cpu()):
        if not bool(valid):
            continue
        offset, owner, length, kind = [int(round(float(row[i]) * scale)) for i, scale in [(12,65536), (13,128), (14,16), (15,4)]]
        if bool(action):
            if kind != 2: raise ReasoningEncodingError("Invalid anchor")
            anchor_owners.append(owner)
            continue
        if kind != (0 if owner == 0 else 1) or not 1 <= length <= 12:
            raise ReasoningEncodingError("Invalid token")
        chunk = bytes(int(round(float(v) * 255)) for v in row[:length])
        if offset in pieces.setdefault(owner, {}): raise ReasoningEncodingError("Duplicate address")
        pieces[owner][offset] = chunk
    if anchor_owners != list(range(1, len(anchor_owners)+1)) or set(pieces) != set(range(len(anchor_owners)+1)):
        raise ReasoningEncodingError("Invalid ownership")
    payloads = []
    for owner in range(len(pieces)):
        data = b""
        for offset, chunk in sorted(pieces[owner].items()):
            if offset != len(data): raise ReasoningEncodingError("Missing byte range")
            data += chunk
        payloads.append(json.loads(data.decode("utf-8")))
    return payloads[0], payloads[1:]

def pretraining_source_hashes(project):
    names = ["00_instance_setup_and_recovery.ipynb", "01_go_rules_and_independent_verifier.ipynb", "02_go_to_csp_and_qcsp.ipynb", "03_curricula_datasets_and_splits.ipynb", "04_native_ahr_architecture.ipynb", "05_csp_qcsp_pretraining.ipynb"]
    result = {}
    for name in names:
        book = nbformat.read(Path(project) / "notebooks" / name, 4)
        result[name] = reasoning_digest([c.source for c in book.cells if c.cell_type == "code" and "ahr-export" in c.metadata.get("tags", [])])
    return result


## Verified curriculum and split discipline

Only six named CSP train/dev shards are opened. Source hashes, row counts, family, split and group isolation are checked first. Derived contexts retain their parent group. A training context enumerates assignments *before* an existential decision and independently labels each residual problem; it does not train the model to choose universal branches. Contexts without both known SAT and known UNSAT actions provide no ranking pair. Cutoffs remain UNKNOWN and are counted separately.

Curriculum order increases variable/constraint count, then observed depth. Optional variable-renaming expansion preserves each original semantic group and its split, while prefix conditioning increases decision contexts. It scales examples within 03's finite curriculum, not the number of independent problems or unbounded problem size. Canonical equivalence checking is limited to five variables. Larger newly generated problem families remain future work.

In [ ]:
CSP_FAMILIES = ("graph_coloring", "boolean_sat", "quantified_sat")

def load_csp_training_rows(project, dataset="starter_v1_0687ec005196"):
    root = Path(project) / "data" / dataset
    manifest = json.loads((root / "manifest.json").read_text())
    shards = {item["file"]: item for item in manifest["shards"]}
    rows = {"train": [], "dev": []}
    receipts = {}
    for split in rows:
        for family in CSP_FAMILIES:
            name = f"{family}_{split}.jsonl"
            item, path = shards[name], root / name
            if file_sha256(path) != item["sha256"] or path.stat().st_size != item["bytes"]:
                raise ValueError(f"Dataset integrity failure: {name}")
            records = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
            if len(records) != item["rows"]: raise ValueError("Row count mismatch")
            for row in records:
                if row["family"] != family or row["split"] != split or row["status"] not in (SAT, UNSAT) or row["rules"] is not None:
                    raise ValueError("Forbidden or incorrectly split pretraining row")
                csp_from_dict(row["problem"])
            rows[split] += records
            receipts[name] = item["sha256"]
    groups = {key: {r["group_id"] for r in value} for key, value in rows.items()}
    if groups["train"] & groups["dev"]: raise ValueError("Group leakage")
    return rows, {"dataset": dataset, "manifest_sha256": file_sha256(root / "manifest.json"), "shards": receipts,
                  "training_route": "csp_only", "trained_on_go": False}

def condition_csp(problem, assignments):
    validate_csp(problem)
    if any(k not in problem.domains or type(v) is not int or v not in problem.domains[k] for k,v in assignments.items()):
        raise ValueError("Invalid prefix assignment")
    constraints = []
    for c in problem.constraints:
        keep = [i for i,name in enumerate(c.scope) if name not in assignments]
        rows = {tuple(row[i] for i in keep) for row in c.allowed if all(name not in assignments or assignments[name] == value for name,value in zip(c.scope,row))}
        constraints.append(TableConstraint(tuple(c.scope[i] for i in keep), tuple(sorted(rows))))
    return CSP({k:v for k,v in problem.domains.items() if k not in assignments}, tuple(constraints), tuple((q,k) for q,k in problem.prefix if k not in assignments))

def verified_residual_status(problem, max_nodes=100000):
    if problem.prefix:
        solved = solve_qcsp(problem, max_nodes=max_nodes)
        reference = reference_qcsp(problem, max_assignments=max_nodes)
        if solved.status == SAT:
            checked = verify_qcsp_strategy(problem, solved.strategy, max_nodes=max_nodes).status
            if checked == UNSAT: raise AssertionError("Invalid QCSP SAT certificate")
            if checked == UNKNOWN: return UNKNOWN
    else:
        solved = solve_csp(problem, max_nodes=max_nodes)
        reference = reference_csp(problem, max_assignments=max_nodes)
        if solved.status == SAT and not verify_assignment(problem, solved.witness):
            raise AssertionError("Invalid CSP witness")
    if UNKNOWN in (solved.status, reference.status): return UNKNOWN
    if solved.status != reference.status: raise AssertionError("Independent solvers disagree")
    return solved.status

def derive_constraint_decisions(rows, max_contexts_per_row=256, max_nodes=100000):
    if type(max_contexts_per_row) is not int or max_contexts_per_row < 1: raise ValueError("Invalid context cap")
    examples, counts = [], Counter()
    for row in rows:
        if row["family"] not in CSP_FAMILIES or row["split"] not in ("train", "dev") or row["status"] not in (SAT, UNSAT):
            raise ValueError("Only known CSP train/dev rows may derive supervision")
        problem = csp_from_dict(row["problem"])
        prefix = problem.prefix or tuple(("exists", name) for name in sorted(problem.domains))
        visited = 0
        for depth, (quantifier, name) in enumerate(prefix):
            if quantifier != "exists": continue
            earlier = [v for _,v in prefix[:depth]]
            for values in itertools.product(*(problem.domains[v] for v in earlier)):
                if visited >= max_contexts_per_row:
                    counts["context_cap_reached"] += 1
                    break
                visited += 1
                observed = dict(zip(earlier,values))
                actions = [{"assign": name, "value": value} for value in problem.domains[name]]
                statuses = [verified_residual_status(condition_csp(problem,{**observed,name:a["value"]}),max_nodes) for a in actions]
                counts.update(statuses)
                if SAT not in statuses or UNSAT not in statuses:
                    counts["no_ranking_pair"] += 1
                    continue
                # Labels and certificates are not part of the neural context/actions.
                context = {"task": "finite_constraint_decision", "problem": csp_to_dict(problem), "observed_prefix": observed, "next_existential": name}
                examples.append({"id": reasoning_digest([row["row_id"],observed,name]), "parent_row": row["row_id"],
                                 "group_id": row["group_id"], "split": row["split"], "family": row["family"],
                                 "difficulty": [len(problem.domains),len(problem.constraints),depth],
                                 "context": context, "actions": actions, "statuses": statuses})
    examples.sort(key=lambda e:(e["difficulty"],e["id"]))
    counts["ranking_contexts"] = len(examples)
    return examples, dict(counts)

def validate_pretraining_examples(examples):
    if not examples: raise ValueError("No ranking examples")
    for row in examples:
        if row["split"] not in ("train", "dev") or row["family"] not in CSP_FAMILIES:
            raise ValueError("Forbidden training provenance")
        if len(row["actions"]) != len(row["statuses"]) or SAT not in row["statuses"] or UNSAT not in row["statuses"]:
            raise ValueError("Invalid ranking labels")
        if set(row["statuses"]) - {SAT,UNSAT,UNKNOWN}: raise ValueError("Invalid status")
        if row["context"].get("task") != "finite_constraint_decision": raise ValueError("Not a CSP context")
        problem = csp_from_dict(row["context"]["problem"])
        prefix = problem.prefix or tuple(("exists",name) for name in sorted(problem.domains))
        observed = row["context"]["observed_prefix"]
        name = row["context"]["next_existential"]
        position = next(i for i,(_,v) in enumerate(prefix) if v == name)
        if prefix[position][0] != "exists" or set(observed) != {v for _,v in prefix[:position]}:
            raise ValueError("Quantifier-prefix mismatch")
        if row["actions"] != [{"assign":name,"value":v} for v in problem.domains[name]]:
            raise ValueError("Action domain mismatch")
    return True

def constraint_margin_loss(output, batch, statuses, margin=0.2):
    energy = output["energies"]["soft"][0,batch["action_mask"][0]]
    if len(energy) != len(statuses) or not bool(torch.isfinite(energy).all()): raise ValueError("Nonfinite/misaligned action energies")
    positive = energy[[i for i,s in enumerate(statuses) if s == SAT]]
    negative = energy[[i for i,s in enumerate(statuses) if s == UNSAT]]
    if not len(positive) or not len(negative): raise ValueError("No verified pair")
    return torch.relu(margin + positive[:,None] - negative[None,:]).mean()


## Deterministic two-GPU training and exact restart

The trainer uses fresh subprocesses, one process per GPU, NCCL and PyTorch DistributedDataParallel. The executable bootstrap lives in this notebook and is passed through `python -c`; no helper source files are created. CPU single-process operation uses the same worker for diagnostics. See the [official DDP documentation](https://docs.pytorch.org/docs/main/generated/torch.nn.parallel.DistributedDataParallel.html).

Each rank consumes its slice of a saved global permutation. The last partial batch wraps deterministically and records that policy. First-epoch order is the curriculum; later epochs shuffle with an epoch-specific generator. FP32 and deterministic operations are used; AMP is disabled and its scaler state is saved. Rank RNG states, model, initial model, optimizer, scheduler, scaler, permutation/cursor/epoch, global step, metrics, source/data/config hashes and distributed world size are checkpointed. There is no replay buffer. Recurrent AHR states reset per example and have no persistent state between optimizer steps.

Rank zero alone owns the run lock. Atomic checkpoints occur at epoch/stage boundaries, at most every 50 steps or 600 seconds. Checkpoint sequence numbers allow another boundary at the same optimizer step; the optimizer step remains separate. Latest two and best are retained. Strict loading rejects a missing, altered or incompatible manifest rather than silently restarting. Exact resume is supported only with the same world size, code, data, configuration and device class. Best selection uses development margin loss only; development action accuracy is conditional finite-constraint accuracy, not a full solver success rate.

In [ ]:
@dataclass(frozen=True)
class CSPTrainConfig:
    seed: int = 20260924
    epochs: int = 12
    batch_per_rank: int = 8
    learning_rate: float = 0.001
    weight_decay: float = 0.0001
    margin: float = 0.2
    checkpoint_steps: int = 50
    checkpoint_seconds: float = 600.0
    world_size: int = 2
    device: str = "cuda"
    purpose: str = "csp_pretraining"

    def __post_init__(self):
        if any(type(v) is not int or v < 1 for v in (self.epochs,self.batch_per_rank,self.world_size,self.checkpoint_steps)):
            raise ValueError("Positive integer training settings required")
        if self.world_size not in (1,2) or self.device not in ("cpu","cuda") or self.device == "cpu" and self.world_size != 1:
            raise ValueError("Supported: one CPU process or one/two GPU processes")
        if self.purpose not in ("csp_pretraining","synthetic_smoke") or not 0 < self.checkpoint_seconds <= 900 or not self.checkpoint_steps <= 250:
            raise ValueError("Invalid checkpoint or purpose setting")
        if not 0 < self.learning_rate <= 0.1 or not 0 <= self.weight_decay <= 1 or not 0 < self.margin <= 10:
            raise ValueError("Invalid optimization setting")

def _safe_run_dir(project, run_id):
    if not isinstance(run_id,str) or not run_id or Path(run_id).name != run_id or run_id in (".",".."):
        raise ValueError("Use a simple run ID")
    return Path(project) / "checkpoints" / run_id

def strict_training_checkpoint(project, run_id, selection="latest"):
    root = _safe_run_dir(project, run_id)
    path = root / "manifest.json"
    if selection not in ("latest","best"): raise ValueError("Choose latest or best")
    if not path.exists():
        if list(root.glob("*.pt")): raise ValueError("Checkpoint exists without committed manifest")
        return None
    manifest = json.loads(path.read_text())
    if manifest.get("schema_version") != 1 or not manifest.get("entries"):
        raise ValueError("Invalid checkpoint manifest")
    entries=manifest["entries"]
    if len({e["file"] for e in entries})!=len(entries) or len({e["global_step"] for e in entries})!=len(entries):
        raise ValueError("Duplicate checkpoint manifest entries")
    if manifest.get("latest")!=max(entries,key=lambda e:e["global_step"])["file"]:
        raise ValueError("Manifest latest would rewind the run")
    states = {}
    for entry in manifest["entries"]:
        value = _checkpoint_entry_state(root,entry)
        if value is None: raise ValueError(f"Corrupt committed checkpoint: {entry.get('file')}")
        states[entry["file"]] = value
    if manifest.get(selection) not in states: raise ValueError("Manifest selection missing")
    state = states[manifest[selection]]
    if state.get("schema") != "ahr_csp_pretraining_v1": raise ValueError("Wrong checkpoint schema")
    return state

def load_pretrained_ahr(project, run_id, device="cpu", selection="best"):
    state = strict_training_checkpoint(project,run_id,selection)
    if state is None: raise FileNotFoundError(f"No pretrained checkpoint in {run_id}")
    meta = state["provenance"]
    if meta.get("training_route") != "csp_only" or meta.get("trained_on_go") is not False or state["config"]["train"]["purpose"] != "csp_pretraining" or state["optimizer_step"] <= 0:
        raise ValueError("Not a trained CSP-only checkpoint; smoke/untrained snapshots are rejected")
    if state["config"]["sources"] != pretraining_source_hashes(project):
        raise ValueError("Source definitions changed since training; explicitly migrate instead of silently loading")
    data_path = Path(project) / state["config"]["data_file"]
    if not data_path.is_file() or file_sha256(data_path) != state["config"]["data_sha256"]:
        raise ValueError("Training data provenance changed")
    model = NativeAHR(AHRConfig(**state["config"]["model"]))
    model.load_state_dict(state["model"],strict=True)
    model.to(device).eval()
    codec = ReasoningCodecConfig(**state["config"]["codec"])
    return model, codec, {**meta, "run_id":run_id,"selection":selection,"purpose":"csp_pretraining",
                          "trained_optimizer_steps":state["optimizer_step"],"source_hashes":state["config"]["sources"],
                          "data_sha256":state["config"]["data_sha256"],"codec_sha256":reasoning_digest(asdict(codec)),
                          "initial_model_state":state["initial_model_state"],"metrics":state["metrics"],
                          "model_sha256":architecture_parameter_digest(model),"configuration":state["config"]}

def validate_prepared_curriculum(data,purpose):
    if data.get("schema")!="ahr_csp_decision_curriculum_v1":raise ValueError("Unexpected curriculum schema")
    provenance=data.get("provenance",{})
    if provenance.get("purpose")!=purpose or provenance.get("training_route")!="csp_only" or provenance.get("trained_on_go") is not False:
        raise ValueError("Curriculum purpose/route mismatch")
    for split in ("train","dev"):
        validate_pretraining_examples(data[split])
        if any(row["split"]!=split for row in data[split]):raise ValueError("Rows in wrong train/dev collection")
    if {r["group_id"] for r in data["train"]}&{r["group_id"] for r in data["dev"]}:raise ValueError("Group leakage")
    return True

def pack_reasoning_examples(examples, codec, device):
    single = [encode_reasoning_actions(e["context"],e["actions"],codec,device) for e in examples]
    length = max(x["x"].shape[1] for x in single)
    batch = {"x":torch.zeros((len(single),length,16),device=device),
             "valid_mask":torch.zeros((len(single),length),device=device,dtype=torch.bool),
             "action_mask":torch.zeros((len(single),length),device=device,dtype=torch.bool)}
    for i,item in enumerate(single):
        size = item["x"].shape[1]
        for key in batch: batch[key][i,:size] = item[key][0]
    return batch

def batched_constraint_loss(output,batch,examples,margin):
    losses,correct = [],0
    for i,row in enumerate(examples):
        energy = output["energies"]["soft"][i,batch["action_mask"][i]]
        if not bool(torch.isfinite(energy).all()) or len(energy) != len(row["statuses"]): raise ValueError("Bad energies")
        pos = energy[[j for j,s in enumerate(row["statuses"]) if s == SAT]]
        neg = energy[[j for j,s in enumerate(row["statuses"]) if s == UNSAT]]
        losses.append(torch.relu(margin+pos[:,None]-neg[None,:]).mean())
        correct += row["statuses"][int(energy.detach().argmin())] == SAT
    return torch.stack(losses).mean(),correct

def evaluate_constraint_ranking(model, examples, codec, batch_size=8, margin=0.2):
    if not examples: return {"contexts":0,"margin_loss":None,"verified_top1_fraction":None}
    was_training = model.training
    model.eval()
    losses,correct = 0.0,0
    with torch.inference_mode():
        for start in range(0,len(examples),batch_size):
            rows=examples[start:start+batch_size]
            batch=pack_reasoning_examples(rows,codec,next(model.parameters()).device)
            loss,count=batched_constraint_loss(model(batch),batch,rows,margin)
            losses+=float(loss)*len(rows); correct+=count
    model.train(was_training)
    return {"contexts":len(examples),"margin_loss":losses/len(examples),"verified_top1_fraction":correct/len(examples)}

def _rank_rng_capture(device):
    return {"python":random.getstate(),"numpy":np.random.get_state(),"torch_cpu":torch.get_rng_state(),
            "torch_cuda":torch.cuda.get_rng_state(device) if device.type == "cuda" else None}

def _rank_rng_restore(rng,device):
    random.setstate(rng["python"]); np.random.set_state(rng["numpy"]); torch.set_rng_state(rng["torch_cpu"])
    if device.type == "cuda": torch.cuda.set_rng_state(rng["torch_cuda"],device)

def _epoch_order(size,seed,epoch):
    if epoch == 0: return list(range(size))
    return torch.randperm(size,generator=torch.Generator().manual_seed(seed+epoch)).tolist()

def _training_worker(project, launch_path, rank):
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel
    from contextlib import nullcontext
    project=Path(project); launch=json.loads(Path(launch_path).read_text()); identity=launch["identity"]
    config=CSPTrainConfig(**identity["train"]); codec=ReasoningCodecConfig(**identity["codec"])
    if identity["sources"] != pretraining_source_hashes(project): raise ValueError("Worker code identity changed")
    if file_sha256(project/identity["data_file"]) != identity["data_sha256"]: raise ValueError("Worker data identity changed")
    prepared=json.loads((project/identity["data_file"]).read_text()); train,dev=prepared["train"],prepared["dev"]
    validate_prepared_curriculum(prepared,config.purpose)
    if {r["group_id"] for r in train}&{r["group_id"] for r in dev}: raise ValueError("Group leakage")
    device=torch.device(f"cuda:{rank}" if config.device=="cuda" else "cpu")
    torch.set_num_threads(1)
    if device.type=="cuda": torch.cuda.set_device(device)
    torch.use_deterministic_algorithms(True); torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
    if config.world_size>1:
        dist.init_process_group("nccl",init_method=f"tcp://127.0.0.1:{launch['port']}",rank=rank,world_size=config.world_size)
    random.seed(config.seed); np.random.seed(config.seed); torch.manual_seed(config.seed)
    model=NativeAHR(AHRConfig(**identity["model"])).to(device)
    initial=architecture_cpu_tree(model.state_dict())
    optimizer=torch.optim.AdamW(model.parameters(),lr=config.learning_rate,weight_decay=config.weight_decay)
    scheduler=torch.optim.lr_scheduler.ExponentialLR(optimizer,gamma=.97)
    scaler=torch.amp.GradScaler("cuda",enabled=False)
    run_dir=_safe_run_dir(project,launch["run_id"])
    lock=checkpoint_lock(run_dir) if rank==0 else nullcontext()
    with lock:
        state=strict_training_checkpoint(project,launch["run_id"])
        epoch,cursor,step,sequence,best=0,0,0,-1,float("inf")
        metrics=[]; order=_epoch_order(len(train),config.seed,0)
        if state is not None:
            if state["config"]!=identity: raise ValueError("Resume configuration/data/code mismatch")
            if state["distributed"]["world_size"]!=config.world_size: raise ValueError("World size mismatch")
            model.load_state_dict(state["model"]); optimizer.load_state_dict(state["optimizer"])
            scheduler.load_state_dict(state["scheduler"]); scaler.load_state_dict(state["scaler"])
            initial=state["initial_model_state"]; epoch=state["sampler"]["epoch"];cursor=state["sampler"]["cursor"]
            order=state["sampler"]["permutation"]; step=state["optimizer_step"];sequence=state["global_step"]
            metrics=state["metrics"];best=state["best_dev_loss"]
            if order!=_epoch_order(len(train),config.seed,epoch) or not 0<=cursor<len(train): raise ValueError("Sampler state invalid")
        wrapped=DistributedDataParallel(model,device_ids=[rank]) if config.world_size>1 else model
        # DDP setup must finish before restoring dropout RNG.
        random.seed(config.seed+rank);np.random.seed(config.seed+rank);torch.manual_seed(config.seed+rank)
        if state is not None: _rank_rng_restore(state["rng"][rank],device)
        resumed_from=step;last_step=step;last_save=time.monotonic()
        def checkpoint(stage,score=None):
            nonlocal sequence,best,last_step,last_save
            rng=_rank_rng_capture(device);rngs=[None]*config.world_size
            if config.world_size>1: dist.all_gather_object(rngs,rng)
            else: rngs=[rng]
            improve=score is not None and score<best
            if improve: best=score
            sequence+=1
            if rank==0:
                payload={"schema":"ahr_csp_pretraining_v1","model":architecture_cpu_tree(model.state_dict()),
                         "initial_model_state":initial,"optimizer":architecture_cpu_tree(optimizer.state_dict()),
                         "scheduler":scheduler.state_dict(),"scaler":scaler.state_dict(),"rng":rngs,
                         "sampler":{"epoch":epoch,"cursor":cursor,"permutation":order,"tail_policy":"deterministic wrap"},
                         "global_step":sequence,"optimizer_step":step,"config":identity,"metrics":metrics,"stage":stage,
                         "best_dev_loss":best,"distributed":{"world_size":config.world_size,"backend":"nccl" if config.world_size>1 else None,"rank_rngs":config.world_size},
                         "replay":None,"provenance":prepared["provenance"]}
                _save_checkpoint_unlocked(run_dir,payload,best=improve,resume_cell="05: Launch / resume (same run ID and configuration)")
                atomic_json(run_dir/"progress.json",{"stage":stage,"optimizer_step":step,"epoch":epoch,"cursor":cursor,"resumed_from":resumed_from,"latest_metrics":metrics[-1] if metrics else None})
            if config.world_size>1: dist.barrier()
            last_step=step;last_save=time.monotonic()
        if state is None:
            baseline=evaluate_constraint_ranking(model,dev,codec,config.batch_per_rank,config.margin)
            metrics.append({"epoch":0,"optimizer_step":0,"development":baseline,"kind":"untrained_baseline"})
            checkpoint("initialized")
        stop_at=launch.get("stop_after_steps")
        while epoch<config.epochs and (stop_at is None or step<stop_at):
            wrapped.train()
            global_batch=config.world_size*config.batch_per_rank
            indices=[order[(cursor+i)%len(order)] for i in range(global_batch)]
            local=indices[rank*config.batch_per_rank:(rank+1)*config.batch_per_rank]
            rows=[train[i] for i in local]
            batch=pack_reasoning_examples(rows,codec,device)
            optimizer.zero_grad(set_to_none=True)
            output=wrapped(batch)
            loss,_=batched_constraint_loss(output,batch,rows,config.margin)
            if not bool(torch.isfinite(loss)): raise FloatingPointError("Nonfinite loss")
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0,error_if_nonfinite=True)
            optimizer.step();step+=1;cursor+=global_batch
            reduced=loss.detach().clone()
            if config.world_size>1:dist.all_reduce(reduced);reduced/=config.world_size
            boundary=cursor>=len(order)
            if boundary:
                epoch+=1;cursor=0;order=_epoch_order(len(train),config.seed,epoch);scheduler.step()
                evaluated=evaluate_constraint_ranking(model,dev,codec,config.batch_per_rank,config.margin)
                metrics.append({"epoch":epoch,"optimizer_step":step,"train_last_batch_loss":float(reduced),"development":evaluated,"kind":"epoch"})
                checkpoint("epoch",evaluated["margin_loss"])
                if rank==0:print(json.dumps(metrics[-1]),flush=True)
            else:
                due=step-last_step>=config.checkpoint_steps or time.monotonic()-last_save>=config.checkpoint_seconds
                if config.world_size>1:
                    flag=torch.tensor(int(due),device=device);dist.all_reduce(flag,op=dist.ReduceOp.MAX);due=bool(flag.item())
                if due:checkpoint("periodic")
        if step!=last_step or epoch>=config.epochs:
            checkpoint("complete" if epoch>=config.epochs else "interrupted_at_requested_step")
        if rank==0:print(json.dumps({"complete":epoch>=config.epochs,"optimizer_step":step,"resumed_from":resumed_from}),flush=True)
    if config.world_size>1:dist.destroy_process_group()

PRETRAIN_BOOTSTRAP = r"""
import sys, pathlib, nbformat
project=pathlib.Path(sys.argv[1]); namespace=globals()
for filename in ['00_instance_setup_and_recovery.ipynb','01_go_rules_and_independent_verifier.ipynb','02_go_to_csp_and_qcsp.ipynb','03_curricula_datasets_and_splits.ipynb','04_native_ahr_architecture.ipynb','05_csp_qcsp_pretraining.ipynb']:
    book=nbformat.read(project/'notebooks'/filename,4)
    for i,cell in enumerate(book.cells):
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
            exec(compile(cell.source,filename+':export-'+str(i),'exec'),namespace)
_training_worker(project,sys.argv[2],int(sys.argv[3]))
"""

def launch_constraint_training(project, run_id, data_file, config=None, codec=None, model_config=None, stop_after_steps=None, require_resume_receipt=True):
    project=Path(project);config=config or CSPTrainConfig();codec=codec or ReasoningCodecConfig()
    model_config=model_config or AHRConfig(max_tokens=codec.max_tokens)
    run_dir=_safe_run_dir(project,run_id);run_dir.mkdir(parents=True,exist_ok=True)
    data_path=Path(data_file)
    if not data_path.is_absolute():data_path=project/data_path
    relative=str(data_path.resolve().relative_to(project.resolve()))
    data=json.loads(data_path.read_text());validate_prepared_curriculum(data,config.purpose)
    if config.device=="cuda" and torch.cuda.device_count()<config.world_size:raise RuntimeError("Requested GPU count unavailable")
    identity={"train":asdict(config),"codec":asdict(codec),"model":asdict(model_config),"sources":pretraining_source_hashes(project),
              "data_file":relative,"data_sha256":file_sha256(data_path),"framework":torch.__version__,"cuda":torch.version.cuda,
              "gpu_classes":[{"name":torch.cuda.get_device_name(i),"capability":list(torch.cuda.get_device_capability(i))} for i in range(config.world_size)] if config.device=="cuda" else []}
    if config.purpose=="synthetic_smoke" and config.epochs*math.ceil(len(data["train"])/(config.world_size*config.batch_per_rank))>8:
        raise ValueError("Recovery smoke must remain short")
    if config.purpose=="csp_pretraining":
        receipt_path=project/"reports/05_exact_training_resume.json"
        receipt=json.loads(receipt_path.read_text()) if receipt_path.exists() else {}
        if receipt.get("status")!="PASS" or receipt.get("sources")!=identity["sources"] or receipt.get("world_size")!=config.world_size or receipt.get("device")!=config.device or receipt.get("model")!=identity["model"] or receipt.get("codec")!=identity["codec"] or receipt.get("framework")!=identity["framework"] or receipt.get("cuda")!=identity["cuda"] or receipt.get("gpu_classes")!=identity["gpu_classes"]:
            raise RuntimeError("Run the matching fresh-process exact-resume smoke before full training")
    with socket.socket() as reservation:
        reservation.bind(("127.0.0.1",0));port=reservation.getsockname()[1]
    launch={"identity":identity,"run_id":run_id,"port":port,"stop_after_steps":stop_after_steps}
    # Per-launch immutable names prevent concurrent launches overwriting a worker's payload.
    launch_file=run_dir/("launch-"+uuid.uuid4().hex+".json");atomic_json(launch_file,launch)
    processes=[];handles=[]
    try:
        for rank in range(config.world_size):
            log_path=run_dir/(launch_file.stem+f"-rank{rank}.log")
            handle=open(log_path,"w");handles.append(handle)
            processes.append(subprocess.Popen([sys.executable,"-c",PRETRAIN_BOOTSTRAP,str(project),str(launch_file),str(rank)],cwd=project,stdout=handle,stderr=subprocess.STDOUT,env={**os.environ,"CUBLAS_WORKSPACE_CONFIG":":4096:8","PYTHONHASHSEED":str(config.seed)}))
        return {"processes":processes,"handles":handles,"run_id":run_id,"launch_file":str(launch_file),"identity":identity}
    except BaseException:
        for process in processes:process.terminate()
        for handle in handles:handle.close()
        raise

def await_constraint_training(job,poll_seconds=1):
    try:
        while any(p.poll() is None for p in job["processes"]):
            if any(p.poll() not in (None,0) for p in job["processes"]):
                raise RuntimeError(f"Training worker failed; inspect {job['launch_file']}")
            time.sleep(poll_seconds)
        if any(p.returncode!=0 for p in job["processes"]):raise RuntimeError(f"Training worker failed; inspect {job['launch_file']}")
    except BaseException:
        for process in job["processes"]:
            if process.poll() is None:process.terminate()
        raise
    finally:
        for handle in job["handles"]:handle.close()
    return {"run_id":job["run_id"],"status":"workers_completed","launch_file":job["launch_file"]}


In [ ]:
def expand_training_variants(rows, copies=1, seed=20260924):
    if type(copies) is not int or not 1<=copies<=16: raise ValueError("Expansion copies must be 1..16")
    expanded=list(rows)
    for row in rows:
        if row["split"]!="train" or row["family"] not in CSP_FAMILIES:raise ValueError("Only training rows may expand")
        original=csp_from_dict(row["problem"])
        for variant in range(1,copies):
            names=list(original.domains)
            rng=random.Random(int(reasoning_digest([seed,row["row_id"],variant])[:16],16))
            renamed=[f"v{variant}_{i}" for i in range(len(names))];rng.shuffle(renamed)
            mapping=dict(zip(names,renamed))
            candidate=CSP({mapping[k]:v for k,v in original.domains.items()},
                          tuple(TableConstraint(tuple(mapping[k] for k in c.scope),c.allowed) for c in original.constraints),
                          tuple((q,mapping[k]) for q,k in original.prefix))
            if canonical_csp(candidate)!=canonical_csp(original):raise AssertionError("Expansion changed group")
            value=copy.deepcopy(row);value["problem"]=csp_to_dict(candidate)
            value["row_id"]=reasoning_digest([row["row_id"],variant,value["problem"]])
            value["provenance"]["renaming_expansion"]={"parent":row["row_id"],"variant":variant,"seed":seed}
            expanded.append(value)
    return expanded

def prepare_csp_curriculum(project, copies=1):
    project=Path(project);rows,provenance=load_csp_training_rows(project)
    source_counts={k:len(v) for k,v in rows.items()}
    rows["train"]=expand_training_variants(rows["train"],copies)
    train,train_counts=derive_constraint_decisions(rows["train"])
    dev,dev_counts=derive_constraint_decisions(rows["dev"])
    validate_pretraining_examples(train);validate_pretraining_examples(dev)
    if {r["group_id"] for r in train}&{r["group_id"] for r in dev}:raise ValueError("Derived group leakage")
    payload={"schema":"ahr_csp_decision_curriculum_v1","train":train,"dev":dev,
             "provenance":{**provenance,"purpose":"csp_pretraining","source_rows":source_counts,"renaming_copies":copies,
                           "train_groups":len({r["group_id"] for r in rows["train"]}),"dev_groups":len({r["group_id"] for r in rows["dev"]}),
                           "derivation_source_hashes":pretraining_source_hashes(project)},
             "counts":{"train":train_counts,"dev":dev_counts}}
    signature=reasoning_digest(payload)
    path=project/"data"/("csp_decisions_"+signature[:12]+".json")
    atomic_json(path,payload)
    return path,payload

def synthetic_resume_curriculum(project):
    records={"train":[],"dev":[]}
    for split,count in (("train",32),("dev",8)):
        for i in range(count):
            name=f"{split}_x{i}"
            problem=CSP({name:(0,1)},(TableConstraint((name,),((i%2,),)),))
            row={"row_id":reasoning_digest([split,i]),"group_id":f"synthetic-{split}-{i}","family":"boolean_sat",
                 "split":split,"status":SAT,"problem":csp_to_dict(problem)}
            derived,_=derive_constraint_decisions([row]);records[split]+=derived
    payload={**records,"schema":"ahr_csp_decision_curriculum_v1",
             "provenance":{"purpose":"synthetic_smoke","training_route":"csp_only","trained_on_go":False,"warning":"Disposable recovery fixtures; not evidence of learned reasoning"}}
    path=Path(project)/"data/05_synthetic_resume_fixtures.json";atomic_json(path,payload)
    return path

def validate_exact_training_resume(project,world_size=2,device="cuda",model_config=None,codec=None):
    project=Path(project);codec=codec or ReasoningCodecConfig();model_config=model_config or AHRConfig(max_tokens=codec.max_tokens)
    config=CSPTrainConfig(epochs=2,batch_per_rank=8,world_size=world_size,device=device,purpose="synthetic_smoke",checkpoint_steps=1)
    data_path=synthetic_resume_curriculum(project)
    suffix=uuid.uuid4().hex[:8];full_id=f"csp05-smoke-full-{suffix}";resumed_id=f"csp05-smoke-resumed-{suffix}"
    full_job=launch_constraint_training(project,full_id,data_path,config,codec,model_config,require_resume_receipt=False)
    await_constraint_training(full_job)
    part_job=launch_constraint_training(project,resumed_id,data_path,config,codec,model_config,stop_after_steps=1,require_resume_receipt=False)
    await_constraint_training(part_job)
    interrupted=strict_training_checkpoint(project,resumed_id)
    assert interrupted["optimizer_step"]==1 and interrupted["sampler"]["cursor"]>0
    resumed_job=launch_constraint_training(project,resumed_id,data_path,config,codec,model_config,require_resume_receipt=False)
    await_constraint_training(resumed_job)
    full=strict_training_checkpoint(project,full_id);resumed=strict_training_checkpoint(project,resumed_id)
    checked=["model","initial_model_state","optimizer","scheduler","scaler","rng","sampler","optimizer_step","config","metrics","distributed","best_dev_loss"]
    for name in checked:
        if not architecture_equal(full[name],resumed[name]):raise AssertionError(f"Fresh-process exact recovery mismatch: {name}")
    progress=json.loads((_safe_run_dir(project,resumed_id)/"progress.json").read_text())
    assert progress["resumed_from"]==1
    try:load_pretrained_ahr(project,resumed_id)
    except ValueError:pass
    else:raise AssertionError("Synthetic smoke incorrectly accepted as pretrained")
    receipt={"status":"PASS","sources":pretraining_source_hashes(project),"world_size":world_size,"device":device,
             "model":asdict(model_config),"codec":asdict(codec),"comparison":"bitwise equality","checked":checked,
             "full_run":full_id,"resumed_run":resumed_id,"interrupted_step":1,"final_optimizer_step":full["optimizer_step"],
             "separate_fresh_process_launches":3,"trained_on_go":False,
             "framework":full["config"]["framework"],"cuda":full["config"]["cuda"],"gpu_classes":full["config"]["gpu_classes"]}
    atomic_json(project/"reports/05_exact_training_resume.json",receipt)
    return receipt


## Validation · safe default Run All

This cell checks round-trip encoding, explicit overflow, visible action values, adversarial quantifier order, UNKNOWN exclusion, input provenance, batched finite loss and model gradients. It prepares all verified decision contexts from the explicit train/dev shards; this is data derivation, with zero parameter updates. The larger configured curriculum can be prepared with `prepare_csp_curriculum(PROJECT, copies=2)`; renamed copies retain their original groups and are not independent new problems.

In [ ]:
torch.set_num_threads(1)
CODEC05=ReasoningCodecConfig()
context={"nested":{"unicode":"Go 棋 / constraints ∀∃","history":[[0,1,-1],[1,0,-1]]},"prefix":[["forall","y"],["exists","x"]]}
actions=[{"assign":"long_variable_name","value":0},{"assign":"long_variable_name","value":1}]
batch=encode_reasoning_actions(context,actions,CODEC05)
assert decode_reasoning_actions(batch)==(context,actions)
anchors=batch["x"][0,batch["action_mask"][0]]
assert not torch.equal(anchors[0,:12],anchors[1,:12]),"Candidate values must reach the anchor"
for limit in [ReasoningCodecConfig(max_tokens=1),ReasoningCodecConfig(max_payload_bytes=1),ReasoningCodecConfig(max_actions=1)]:
    try:encode_reasoning_actions(context,actions,limit)
    except ReasoningEncodingError:pass
    else:raise AssertionError("Overflow silently accepted")
equality=TableConstraint(("x","y"),((0,0),(1,1)))
ex_all=CSP({"x":(0,1),"y":(0,1)},(equality,),(("exists","x"),("forall","y")))
all_ex=CSP(ex_all.domains,(equality,),(("forall","y"),("exists","x")))
assert verified_residual_status(ex_all)==UNSAT and verified_residual_status(all_ex)==SAT
assert verified_residual_status(all_ex,max_nodes=0)==UNKNOWN
fixture=lambda problem:{"family":"quantified_sat","split":"train","status":SAT,"row_id":reasoning_digest(csp_to_dict(problem)),"group_id":"quantifier-fixture","problem":csp_to_dict(problem)}
quantified,_=derive_constraint_decisions([fixture(all_ex)])
assert len(quantified)==2
assert {(e["context"]["observed_prefix"]["y"],e["statuses"].index(SAT)) for e in quantified}=={(0,0),(1,1)}
no_choice,_=derive_constraint_decisions([fixture(ex_all)])
assert not no_choice
unknown_examples,unknown_counts=derive_constraint_decisions([fixture(all_ex)],max_nodes=0)
assert not unknown_examples and unknown_counts[UNKNOWN]>0
CURRICULUM05,PREPARED05=prepare_csp_curriculum(PROJECT,copies=1)
model05=NativeAHR(AHRConfig(max_tokens=CODEC05.max_tokens))
probe_rows=PREPARED05["train"][:3]
probe_batch=pack_reasoning_examples(probe_rows,CODEC05,"cpu")
probe_output=model05(probe_batch)
probe_loss,_=batched_constraint_loss(probe_output,probe_batch,probe_rows,.2)
assert torch.isfinite(probe_loss)
probe_loss.backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model05.parameters())
validation05={"status":"PASS","optimizer_updates":0,"source_hashes":pretraining_source_hashes(PROJECT),
              "codec_round_trip":True,"quantifier_order":True,"unknown_excluded":True,"all_parameter_gradients_finite":True,
              "curriculum_file":str(CURRICULUM05.relative_to(PROJECT)),"curriculum_sha256":file_sha256(CURRICULUM05),
              "counts":PREPARED05["counts"],"provenance":PREPARED05["provenance"],
              "training_status":"NOT_RUN_IN_DEFAULT_VALIDATION","visual_review":"saved outputs inspected; no graphical notebook renderer available"}
atomic_json(PROJECT/"reports/05_pretraining_validation.json",validation05)
print(json.dumps({k:validation05[k] for k in ["status","optimizer_updates","curriculum_file","counts"]},indent=2))

## Launch / resume · explicit training mode

Default `AHR05_MODE=validate` stops after validation. Following authorization for training, use a fresh project kernel with `AHR05_MODE=smoke` to run four real optimizer steps per control run on disposable fixtures (eight total updates across the uninterrupted control and interrupted/resumed run). The checkpoint comparison must pass before training. Then run with `AHR05_MODE=train` and a unique `AHR05_RUN_ID`; defaults train 12 full epochs on two GPUs, eight examples per rank per step. The receipt must match current exported source, model, codec and distributed configuration. Do not edit exported cells mid-run.

Exact commands from the workspace (environment assignments are launch settings, not helper scripts):

```text
AHR05_MODE=smoke .venv/bin/python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=ahr-go --ExecutePreprocessor.timeout=3600 notebooks/05_csp_qcsp_pretraining.ipynb
AHR05_MODE=train AHR05_RUN_ID=csp05-initial-12epochs .venv/bin/python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=ahr-go --ExecutePreprocessor.timeout=3600 notebooks/05_csp_qcsp_pretraining.ipynb
```

To **resume**, rerun the *second exact command with the same run ID*, same data and same exported definitions. The saved rank RNGs and permutation/cursor restore automatically; `progress.json` records the resumed optimizer step. Never delete a broken manifest to restart. If code/configuration changes, use a new run ID and revalidate recovery. A completed run is detected and produces its final report without additional optimizer updates. The loader defaults to the best *trained* development checkpoint; original initialization is preserved for notebook 07's matched frozen control.

Expected initial run: several minutes, machine-dependent. A process killed during an optimizer step resumes from the most recent committed boundary, replaying that unsaved work. One optimizer step is the indivisible checkpoint unit. The workspace is on the persistent boot disk, but VM deletion policy is unverified. Before deleting/recycling, copy best/final checkpoints and their manifest plus data/notebooks off-instance and verify SHA-256 hashes.

In [ ]:
MODE05=os.environ.get("AHR05_MODE","validate")
if MODE05=="smoke":
    print(json.dumps(validate_exact_training_resume(PROJECT),indent=2))
elif MODE05=="train":
    RUN05=os.environ.get("AHR05_RUN_ID","csp05-initial-12epochs")
    train_config05=CSPTrainConfig()
    training_job05=launch_constraint_training(PROJECT,RUN05,CURRICULUM05,train_config05,CODEC05)
    print("Training launched:",RUN05,flush=True)
    print(await_constraint_training(training_job05),flush=True)
    trained05,codec05,meta05=load_pretrained_ahr(PROJECT,RUN05,device="cuda:0")
    best_dev05=evaluate_constraint_ranking(trained05,PREPARED05["dev"],codec05)
    latest05=strict_training_checkpoint(PROJECT,RUN05)
    training_report05={"status":"COMPLETE" if latest05["sampler"]["epoch"]==train_config05.epochs else "INCOMPLETE",
                       "run_id":RUN05,"epochs":latest05["sampler"]["epoch"],"optimizer_steps":latest05["optimizer_step"],
                       "best_trained_step":meta05["trained_optimizer_steps"],"best_development":best_dev05,
                       "metrics":latest05["metrics"],"model_sha256":meta05["model_sha256"],"provenance":latest05["provenance"],
                       "world_size":train_config05.world_size,"training_route":"csp_only","trained_on_go":False,
                       "sources":pretraining_source_hashes(PROJECT),"claim":"conditional finite-constraint action ranking; Go transfer requires 07"}
    atomic_json(PROJECT/"reports/05_training_results.json",training_report05)
    print(json.dumps(training_report05,indent=2))
elif MODE05!="validate":
    raise ValueError("AHR05_MODE must be validate, smoke or train")
else:
    print("Validation complete. Full training is an explicit separate launch; no optimizer updates in this mode.")

## Complete bounded constraint solving · frozen development evaluation

The executed [notebook 05 solving companion](../reports/05_constraint_solving_evaluation.ipynb) measures complete CSP/QCSP solving on the 14 explicit development rows (7 independent groups), using the trained checkpoint and its preserved initialization. It contains the bounded AHR energy search adapter, independent certificate checks, quantifier-order tests and cutoff tests. It creates no optimizer and opens no final-test or Go data. Restart its project kernel and Run All to repeat this short forward-only evaluation.

Both models correctly resolve all 14 development problems (10 SAT, 4 UNSAT), using **109 search nodes and 49 model calls each**. This establishes functioning bounded solving on these small examples; it shows **no search-efficiency improvement from training** in this evaluation. Development data was also used to select the checkpoint. See the inspectable [JSON results](../reports/05_constraint_solving_results.json) for certificates, provenance and per-problem measurements.